In [1]:
# -----------------------------
# 0. Libraries
# -----------------------------
import pandas as pd
import numpy as np
from prophet import Prophet
from tqdm import tqdm

# -----------------------------
# 1. Load Data
# -----------------------------
df = pd.read_csv("simple_synthetic_sales_data.csv")
df["date"] = pd.to_datetime(df["date"])

# Filter Books
books_df = df[df["category"] == "Books"].copy()

# -----------------------------
# 2. Forecast Function
# -----------------------------
def forecast_with_intervals(group, horizon=7):
    temp = group.sort_values("date").copy()
    
    temp = temp.rename(columns={
        "date": "ds",
        "sales": "y"
    })
    
    # Prophet model (95% interval default)
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        interval_width=0.95
    )
    
    model.add_regressor("promo")
    model.add_regressor("price")
    
    model.fit(temp[["ds", "y", "promo", "price"]])
    
    # Future
    future = model.make_future_dataframe(periods=horizon)
    
    future = future.merge(
        temp[["ds", "promo", "price"]],
        on="ds",
        how="left"
    )
    
    future["promo"] = future["promo"].fillna(0)
    future["price"] = future["price"].fillna(method="ffill")
    
    forecast = model.predict(future)
    
    future_fc = forecast.tail(horizon)
    
    # -----------------------------
    # 95% interval
    # -----------------------------
    cum_mean_95 = future_fc["yhat"].sum()
    cum_lower_95 = future_fc["yhat_lower"].sum()
    cum_upper_95 = future_fc["yhat_upper"].sum()
    
    # -----------------------------
    # 80% interval (refit model)
    # -----------------------------
    model_80 = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        interval_width=0.80
    )
    
    model_80.add_regressor("promo")
    model_80.add_regressor("price")
    
    model_80.fit(temp[["ds", "y", "promo", "price"]])
    
    forecast_80 = model_80.predict(future)
    future_fc_80 = forecast_80.tail(horizon)
    
    cum_lower_80 = future_fc_80["yhat_lower"].sum()
    cum_upper_80 = future_fc_80["yhat_upper"].sum()
    
    return {
        "forecast_mean": cum_mean_95,
        "lower_95": cum_lower_95,
        "upper_95": cum_upper_95,
        "lower_80": cum_lower_80,
        "upper_80": cum_upper_80
    }

# -----------------------------
# 3. Run for all products
# -----------------------------
results = []

for product, group in tqdm(books_df.groupby("product")):
    try:
        res = forecast_with_intervals(group)
        
        results.append({
            "product": product,
            **res
        })
        
    except Exception as e:
        print(f"Error: {product} -> {e}")

results_df = pd.DataFrame(results)

# -----------------------------
# 4. Rank Top-3
# -----------------------------
top3 = results_df.sort_values("forecast_mean", ascending=False).head(3)

# -----------------------------
# 5. Output
# -----------------------------
print("\nTop-3 Products in Books (7-day cumulative forecast):\n")
print(top3.round(2))


  0%|                                                                                                                                                                                        | 0/6 [00:00<?, ?it/s]19:53:58 - cmdstanpy - INFO - Chain [1] start processing
19:53:58 - cmdstanpy - INFO - Chain [1] done processing
/var/folders/kp/02w47g1j4479m7fhr1vcd4g40000gn/T/ipykernel_1200/2637709264.py:51: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  future["price"] = future["price"].fillna(method="ffill")
19:53:58 - cmdstanpy - INFO - Chain [1] start processing
19:53:58 - cmdstanpy - INFO - Chain [1] done processing
 17%|█████████████████████████████▎                                                                                                                                                  | 1/6 [00:00<00:01,  3.96it/s]19:53:59 - cmdstanpy - INFO - Chain [1] start processing
19:53:59 - cmdstanpy 


Top-3 Products in Books (7-day cumulative forecast):

     product  forecast_mean  lower_95  upper_95  lower_80  upper_80
1     Comics        1518.29   1316.17   1723.50   1378.47   1657.93
3      Novel        1493.79   1273.08   1714.25   1350.51   1637.93
0  Biography        1353.44   1132.82   1577.99   1206.11   1495.62
